# 🩻 ChestNet — Custom CNN from Scratch (Full A→Z Pipeline)

**Dataset assumed:** NIH ChestX-ray14 (`Data_Entry_2017.csv` + `images_001 … images_012` folders) — **multi-label** (one X-ray can have several diseases).
**Framework:** PyTorch · **Input:** grayscale 224×224

| Step | What it does |
|---|---|
| 0 | Install / imports |
| 1 | Config ⚙️ |
| 2 | Helper functions 🧰 |
| 3 | Resize images (run once) 🖼️ |
| 4 | Patient-level train/val/test split (run once) ✂️ |
| 5 | Compute mean/std from train set (run once) 📊 |
| 6 | Dataset + DataLoaders 🗂️ |
| 7 | Custom CNN model 🧠 |
| 8 | Sanity check ✅ |
| 9 | Training 🏋️ |
| 10 | Evaluation on test set 📈 |
| 11 | Single-image prediction + Grad-CAM 🔥 |

> 💡 Run cells **top to bottom**. Steps 3–5 only need to run **once** — after that you can skip them.

## 0️⃣ Install & imports

In [ ]:
import json
import random
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms as T

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,
                             precision_score, recall_score, f1_score,
                             multilabel_confusion_matrix)

print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 1️⃣ Config ⚙️
Change paths and hyperparameters here only.

In [ ]:
# ---------- Paths ----------
from pathlib import Path

# ---------- Paths ----------
ROOT = Path("/content/drive/MyDrive/ChestNet")
DATA_DIR = ROOT / "data"
RAW_IMAGE_DIR = DATA_DIR / "raw_images"          # put images_001 ... images_012 folders here
IMAGE_DIR = DATA_DIR / "images_256"              # resized copies are written here (step 3)
CSV_PATH = DATA_DIR / "Data_Entry_2017.csv"      # NIH metadata file
SPLIT_DIR = DATA_DIR / "splits"                  # train/val/test CSVs (step 4)
STATS_PATH = DATA_DIR / "norm_stats.json"        # mean/std (step 5)
OUT_DIR = ROOT / "outputs" / "custom_cnn"        # checkpoints, plots, reports
# ---------- Labels (14 NIH diseases) ----------
CLASSES = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass",
    "Nodule", "Pneumonia", "Pneumothorax", "Consolidation", "Edema",
    "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia",
]
NUM_CLASSES = len(CLASSES)

# ---------- Image ----------
RESIZE_TO = 256      # size stored on disk (step 3)
IMG_SIZE = 224       # size fed to the network
IN_CHANNELS = 1      # grayscale X-rays

# ---------- Split ----------
TEST_FRAC = 0.20
VAL_FRAC = 0.10
SEED = 42

# ---------- Training ----------
BATCH_SIZE = 32
EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 7               # early-stopping patience (epochs)
MAX_POS_WEIGHT = 50.0      # cap for class-imbalance weights

# ⚠️ In Jupyter on Windows, num_workers > 0 usually crashes (classes defined in a
# notebook can't be sent to worker processes). Keep 0 in the notebook.
NUM_WORKERS = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"  # mixed precision only on GPU

OUT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)

## 2️⃣ Helper functions 🧰

In [ ]:
def set_seed(seed: int = SEED):
    """Make results repeatable."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True  # faster convs for fixed image size


def load_stats():
    with open(STATS_PATH) as f:
        s = json.load(f)
    return s["mean"], s["std"]


def compute_aucs(labels: np.ndarray, probs: np.ndarray) -> np.ndarray:
    """ROC-AUC per class. NaN if a class has only one label value in this set."""
    aucs = []
    for i in range(labels.shape[1]):
        if len(np.unique(labels[:, i])) < 2:
            aucs.append(np.nan)
        else:
            aucs.append(roc_auc_score(labels[:, i], probs[:, i]))
    return np.array(aucs)


def plot_history(history: dict, save_path=None, show=False):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(epochs, history["train_loss"], label="train")
    ax[0].plot(epochs, history["val_loss"], label="val")
    ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(epochs, history["train_auc"], label="train")
    ax[1].plot(epochs, history["val_auc"], label="val")
    ax[1].set_title("Mean ROC-AUC"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120)
    plt.show() if show else plt.close()

## 3️⃣ Resize images 🖼️ (run once)
Resizes the raw 1024×1024 PNGs to 256×256 into **one flat folder** → much faster training.

In [ ]:
def resize_one(src):
    dst = IMAGE_DIR / src.name
    if dst.exists():
        return
    img = Image.open(src).convert("L")
    img = img.resize((RESIZE_TO, RESIZE_TO), Image.Resampling.LANCZOS)
    img.save(dst)


def resize_all_images():
    IMAGE_DIR.mkdir(parents=True, exist_ok=True)
    files = list(RAW_IMAGE_DIR.rglob("*.png"))   # searches images_001/images, images_002/images, ...
    print(f"Found {len(files)} images")
    with ThreadPoolExecutor(max_workers=8) as ex:
        list(tqdm(ex.map(resize_one, files), total=len(files)))
    print("Done ✅ ->", IMAGE_DIR)


resize_all_images()

## 4️⃣ Patient-level split ✂️ (run once)
All images of one patient go into **only one** split → prevents data leakage.

In [ ]:
print("ROOT:", ROOT, "| exists:", ROOT.exists())
print("CSV_PATH:", CSV_PATH, "| exists:", CSV_PATH.exists())
print("RAW_IMAGE_DIR:", RAW_IMAGE_DIR, "| exists:", RAW_IMAGE_DIR.exists())
print("IMAGE_DIR:", IMAGE_DIR, "| exists:", IMAGE_DIR.exists())
print("PNGs in IMAGE_DIR:", len(list(IMAGE_DIR.glob("*.png"))) if IMAGE_DIR.exists() else 0)
print("Contents of DATA_DIR:", [p.name for p in DATA_DIR.iterdir()] if DATA_DIR.exists() else "missing")

In [ ]:
import zipfile, time
t = time.time()
zip_path = DATA_DIR / "The NIH ChestX-ray14 dataset.zip"

with zipfile.ZipFile(zip_path) as z:
    print("Files in zip:", len(z.namelist()))
    print(z.namelist()[:10])          # peek at the structure
    z.extractall("/content/raw_images")

print("Done in", round(time.time() - t), "s")

In [ ]:
RAW_IMAGE_DIR = Path("/content/raw_images")

In [ ]:
!cp -r /content/drive/MyDrive/ChestNet/data/images_256 /content/images_256

In [ ]:
from pathlib import Path

# 1. Did extraction happen?
raw = Path("/content/raw_images")
print("Extracted folder exists:", raw.exists())
if raw.exists():
    pngs = list(raw.rglob("*.png"))
    print("PNGs found after extraction:", len(pngs))
    print("Example:", pngs[:2])

# 2. Is images_256 still empty?
print("PNGs in IMAGE_DIR:", len(list(IMAGE_DIR.glob("*.png"))))

In [ ]:
from pathlib import Path
from PIL import Image
from tqdm import tqdm

RAW_IMAGE_DIR = Path("/content/raw_images")   # extracted images live under here
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

src_files = list(RAW_IMAGE_DIR.rglob("*.png"))
print("Source images:", len(src_files))

for p in tqdm(src_files):
    out = IMAGE_DIR / p.name
    if out.exists():          # skip already-resized files if you re-run
        continue
    img = Image.open(p).convert("L")                 # grayscale, matches IN_CHANNELS = 1
    img = img.resize((RESIZE_TO, RESIZE_TO), Image.BILINEAR)
    img.save(out)

print("PNGs in IMAGE_DIR:", len(list(IMAGE_DIR.glob("*.png"))))

In [ ]:
def prepare_splits():
    df = pd.read_csv(CSV_PATH)[["Image Index", "Finding Labels", "Patient ID"]]

    # Keep only images that exist on disk
    existing = {p.name for p in IMAGE_DIR.glob("*.png")}
    df = df[df["Image Index"].isin(existing)].reset_index(drop=True)
    print(f"Images with metadata on disk: {len(df)}")

    # "Effusion|Mass" -> multi-hot columns. "No Finding" = all zeros.
    split_labels = df["Finding Labels"].str.split("|")
    for c in CLASSES:
        df[c] = split_labels.apply(lambda labels: int(c in labels))

    # ---- Patient-level split ----
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_FRAC, random_state=SEED)
    trainval_idx, test_idx = next(gss.split(df, groups=df["Patient ID"]))
    trainval, test = df.iloc[trainval_idx], df.iloc[test_idx]

    val_size = VAL_FRAC / (1 - TEST_FRAC)
    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=SEED)
    train_idx, val_idx = next(gss2.split(trainval, groups=trainval["Patient ID"]))
    train, val = trainval.iloc[train_idx], trainval.iloc[val_idx]

    # Leakage check
    tr, va, te = set(train["Patient ID"]), set(val["Patient ID"]), set(test["Patient ID"])
    assert not (tr & va) and not (tr & te) and not (va & te), "Patient overlap found!"

    for name, part in [("train", train), ("val", val), ("test", test)]:
        part.to_csv(SPLIT_DIR / f"{name}.csv", index=False)
        print(f"{name:5s}: {len(part):6d} images | {part['Patient ID'].nunique():5d} patients")

    print("\nPositive count per class (train):")
    print(train[CLASSES].sum().sort_values(ascending=False).to_string())


prepare_splits()

## 5️⃣ Compute mean / std 📊 (run once)
Uses the **train set only** — using val/test here would leak information.

In [ ]:
def compute_stats(max_images=5000):
    tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor()])
    ds = ChestXrayDataset(SPLIT_DIR / "train.csv", transform=tf)
    g = torch.Generator().manual_seed(SEED)
    idx = torch.randperm(len(ds), generator=g)[:max_images].tolist()
    loader = DataLoader(Subset(ds, idx), batch_size=64, num_workers=NUM_WORKERS)

    total, sq_total, n_pixels = 0.0, 0.0, 0
    for x, _ in tqdm(loader):
        total += x.sum().item()
        sq_total += (x ** 2).sum().item()
        n_pixels += x.numel()

    mean = total / n_pixels
    std = (sq_total / n_pixels - mean ** 2) ** 0.5
    with open(STATS_PATH, "w") as f:
        json.dump({"mean": mean, "std": std}, f, indent=2)
    print(f"mean={mean:.4f} std={std:.4f} saved ✅")

## 6️⃣ Dataset + DataLoaders 🗂️
> 💡 The cell right after this one runs `compute_stats()` from step 5 (it needs this Dataset class first).

In [ ]:
class ChestXrayDataset(Dataset):
    """Returns (image_tensor[1,H,W], multi_hot_label[14])."""

    def __init__(self, csv_path, image_dir=IMAGE_DIR, transform=None):
        df = pd.read_csv(csv_path)
        self.paths = df["Image Index"].values
        self.labels = df[CLASSES].values.astype("float32")
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_dir / self.paths[idx]).convert("L")  # grayscale
        if self.transform:
            img = self.transform(img)
        return img, torch.from_numpy(self.labels[idx])


def get_transforms(train: bool, mean, std):
    # ⚠️ No horizontal flip: the heart is on the left, so flipping creates unrealistic anatomy.
    if train:
        return T.Compose([
            T.RandomResizedCrop(IMG_SIZE, scale=(0.80, 1.0), ratio=(0.95, 1.05)),
            T.RandomRotation(degrees=7),
            T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
            T.ColorJitter(brightness=0.15, contrast=0.15),
            T.ToTensor(),
            T.Normalize([mean], [std]),
        ])
    return T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize([mean], [std]),
    ])


def get_loaders():
    mean, std = load_stats()
    train_ds = ChestXrayDataset(SPLIT_DIR / "train.csv", transform=get_transforms(True, mean, std))
    val_ds = ChestXrayDataset(SPLIT_DIR / "val.csv", transform=get_transforms(False, mean, std))
    test_ds = ChestXrayDataset(SPLIT_DIR / "test.csv", transform=get_transforms(False, mean, std))

    common = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                  pin_memory=(DEVICE == "cuda"), persistent_workers=NUM_WORKERS > 0)
    train_loader = DataLoader(train_ds, shuffle=True, drop_last=True, **common)
    val_loader = DataLoader(val_ds, shuffle=False, **common)
    test_loader = DataLoader(test_ds, shuffle=False, **common)
    return train_loader, val_loader, test_loader

In [ ]:
# 📊 Step 5 — run once (needs ChestXrayDataset from the cell above)
compute_stats()

In [ ]:
train_loader, val_loader, test_loader = get_loaders()
print(f"Batches -> train {len(train_loader)}, val {len(val_loader)}, test {len(test_loader)}")

## 7️⃣ Custom CNN model 🧠 (from scratch)
Residual blocks (conv → BN → ReLU → conv → BN + shortcut) let a from-scratch network train deeper without vanishing gradients.

In [ ]:
class ResidualBlock(nn.Module):
    """conv-BN-ReLU-conv-BN + shortcut."""

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)

        # 1x1 conv to match shape when channels or resolution change
        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + self.shortcut(x))


class ChestNetCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=14, dropout=0.3):
        super().__init__()
        # Stem: 224 -> 112 -> 56
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 32, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.stage1 = nn.Sequential(ResidualBlock(32, 64, 1), ResidualBlock(64, 64))       # 56x56
        self.stage2 = nn.Sequential(ResidualBlock(64, 128, 2), ResidualBlock(128, 128))    # 28x28
        self.stage3 = nn.Sequential(ResidualBlock(128, 256, 2), ResidualBlock(256, 256))   # 14x14
        self.stage4 = nn.Sequential(ResidualBlock(256, 512, 2), ResidualBlock(512, 512))   # 7x7

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(512, num_classes)   # raw logits (no sigmoid here!)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01); nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = torch.flatten(self.pool(x), 1)
        return self.fc(self.dropout(x))


_m = ChestNetCNN(IN_CHANNELS, NUM_CLASSES)
print(_m(torch.randn(2, IN_CHANNELS, IMG_SIZE, IMG_SIZE)).shape)  # -> torch.Size([2, 14])
print(f"Params: {sum(p.numel() for p in _m.parameters()) / 1e6:.2f} M")
del _m

## 8️⃣ Sanity check ✅
Check shapes, value ranges and labels **before** spending hours training.

In [ ]:
x, y = next(iter(train_loader))
print("Image batch:", x.shape, x.dtype, f"min={x.min():.2f} max={x.max():.2f} mean={x.mean():.2f}")
print("Label batch:", y.shape, "| positives in batch:", int(y.sum()))

_model = ChestNetCNN(IN_CHANNELS, NUM_CLASSES).to(DEVICE)
with torch.no_grad():
    print("Model output:", _model(x.to(DEVICE)).shape)  # [B, 14]
del _model

# Show augmented samples (un-normalized)
mean, std = load_stats()
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    img = x[i, 0] * std + mean
    names = [CLASSES[j] for j in torch.where(y[i] == 1)[0]] or ["No Finding"]
    ax.imshow(img, cmap="gray"); ax.set_title(", ".join(names), fontsize=8); ax.axis("off")
plt.tight_layout(); plt.savefig(OUT_DIR / "sample_batch.png", dpi=100); plt.show()

## 9️⃣ Training 🏋️
- **Loss:** `BCEWithLogitsLoss` with `pos_weight` → rare diseases count more
- **Optimizer:** AdamW · **LR schedule:** ReduceLROnPlateau on val AUC
- **Early stopping** + best checkpoint saved by **val mean AUC**

In [ ]:
def get_pos_weight():
    """neg/pos ratio per class -> rare diseases get more weight in the loss."""
    train = pd.read_csv(SPLIT_DIR / "train.csv")
    pos = train[CLASSES].sum().values
    neg = len(train) - pos
    w = np.clip(neg / np.maximum(pos, 1), 1.0, MAX_POS_WEIGHT)
    return torch.tensor(w, dtype=torch.float32)


def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    train = optimizer is not None
    model.train(train)
    total_loss, probs_list, labels_list = 0.0, [], []

    with torch.set_grad_enabled(train):
        for x, y in tqdm(loader, leave=False, desc="train" if train else "val"):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            with torch.autocast(device_type=DEVICE, enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y)

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)  # stability
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item() * x.size(0)
            probs_list.append(torch.sigmoid(logits.float()).detach().cpu())
            labels_list.append(y.cpu())

    probs = torch.cat(probs_list).numpy()
    labels = torch.cat(labels_list).numpy()
    aucs = compute_aucs(labels, probs)
    return total_loss / len(loader.dataset), float(np.nanmean(aucs)), aucs


def train_model():
    set_seed()
    model = ChestNetCNN(IN_CHANNELS, NUM_CLASSES).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=get_pos_weight().to(DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.3, patience=2)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    mean, std = load_stats()
    history = {k: [] for k in ["train_loss", "val_loss", "train_auc", "val_auc", "lr"]}
    best_auc, bad_epochs = 0.0, 0

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_auc, _ = run_epoch(model, train_loader, criterion, optimizer, scaler)
        va_loss, va_auc, va_aucs = run_epoch(model, val_loader, criterion)
        scheduler.step(va_auc)

        lr = optimizer.param_groups[0]["lr"]
        for k, v in zip(history, [tr_loss, va_loss, tr_auc, va_auc, lr]):
            history[k].append(v)

        print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} AUC {tr_auc:.4f} | "
              f"val loss {va_loss:.4f} AUC {va_auc:.4f} | lr {lr:.1e} | {time.time() - t0:.0f}s")

        if va_auc > best_auc:
            best_auc, bad_epochs = va_auc, 0
            torch.save({
                "model_state": model.state_dict(),
                "epoch": epoch,
                "val_auc": float(va_auc),
                "classes": CLASSES,
                "mean": float(mean), "std": float(std),
            }, OUT_DIR / "best_model.pt")
            print(f"  💾 New best (val AUC {best_auc:.4f})")
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print("  ⏹️ Early stopping")
                break

        pd.DataFrame(history).to_csv(OUT_DIR / "history.csv", index=False)
        plot_history(history, OUT_DIR / "training_curves.png")

    print(f"\nBest val mean AUC: {best_auc:.4f}")
    return history


history = train_model()
plot_history(history, OUT_DIR / "training_curves.png", show=True)

## 🔟 Evaluation on the test set 📈
Thresholds are tuned on **validation** data, then applied to **test** (picking thresholds on test = cheating 🚫).

In [ ]:
@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    probs, labels = [], []
    for x, y in tqdm(loader, leave=False):
        logits = model(x.to(DEVICE))
        probs.append(torch.sigmoid(logits).cpu()); labels.append(y)
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()


def best_thresholds(labels, probs):
    """Per-class threshold that maximizes F1 on VALIDATION data."""
    th = []
    for i in range(labels.shape[1]):
        p, r, t = precision_recall_curve(labels[:, i], probs[:, i])
        f1 = 2 * p * r / (p + r + 1e-8)
        th.append(t[np.argmax(f1[:-1])] if len(t) else 0.5)
    return np.array(th)


ckpt = torch.load(OUT_DIR / "best_model.pt", map_location=DEVICE)
model = ChestNetCNN(IN_CHANNELS, NUM_CLASSES).to(DEVICE)
model.load_state_dict(ckpt["model_state"])
print(f"Loaded epoch {ckpt['epoch']} (val AUC {ckpt['val_auc']:.4f})")

val_probs, val_labels = predict_loader(model, val_loader)
thresholds = best_thresholds(val_labels, val_probs)
np.save(OUT_DIR / "thresholds.npy", thresholds)

test_probs, test_labels = predict_loader(model, test_loader)
test_preds = (test_probs >= thresholds).astype(int)
aucs = compute_aucs(test_labels, test_probs)
cms = multilabel_confusion_matrix(test_labels, test_preds)

rows = []
for i, c in enumerate(CLASSES):
    tn, fp, fn, tp = cms[i].ravel()
    rows.append({
        "class": c, "AUC": aucs[i], "threshold": thresholds[i],
        "precision": precision_score(test_labels[:, i], test_preds[:, i], zero_division=0),
        "recall(sensitivity)": recall_score(test_labels[:, i], test_preds[:, i], zero_division=0),
        "specificity": tn / (tn + fp + 1e-8),
        "F1": f1_score(test_labels[:, i], test_preds[:, i], zero_division=0),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn, "support": int(test_labels[:, i].sum()),
    })
report = pd.DataFrame(rows)
report.loc[len(report)] = {"class": "MEAN", "AUC": np.nanmean(aucs),
                           "precision": report["precision"].mean(),
                           "recall(sensitivity)": report["recall(sensitivity)"].mean(),
                           "specificity": report["specificity"].mean(), "F1": report["F1"].mean()}
report.to_csv(OUT_DIR / "test_report.csv", index=False)
report.round(3)

In [ ]:
# 📉 ROC curves
plt.figure(figsize=(9, 8))
for i, c in enumerate(CLASSES):
    fpr, tpr, _ = roc_curve(test_labels[:, i], test_probs[:, i])
    plt.plot(fpr, tpr, label=f"{c} ({aucs[i]:.2f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC curves - Test set"); plt.legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT_DIR / "roc_curves.png", dpi=120); plt.show()

In [ ]:
# 🟦 Confusion matrix per disease
fig, axes = plt.subplots(3, 5, figsize=(18, 11))
for i, ax in enumerate(axes.flat):
    if i >= NUM_CLASSES:
        ax.axis("off"); continue
    ax.imshow(cms[i], cmap="Blues")
    for (r, k), v in np.ndenumerate(cms[i]):
        ax.text(k, r, v, ha="center", va="center")
    ax.set_title(CLASSES[i], fontsize=9)
    ax.set_xticks([0, 1], ["Pred 0", "Pred 1"]); ax.set_yticks([0, 1], ["True 0", "True 1"])
plt.tight_layout(); plt.savefig(OUT_DIR / "confusion_matrices.png", dpi=110); plt.show()

## 1️⃣1️⃣ Single-image prediction + Grad-CAM 🔥
Grad-CAM shows **where** the model looked for a disease (good sanity check for medical AI).

In [ ]:
def grad_cam(model, x, class_idx):
    store = {}

    def fwd_hook(_, __, out):
        store["act"] = out
        out.register_hook(lambda g: store.__setitem__("grad", g))

    h = model.stage4.register_forward_hook(fwd_hook)
    logits = model(x)
    model.zero_grad()
    logits[0, class_idx].backward()
    h.remove()

    act, grad = store["act"][0], store["grad"][0]           # [C, H, W]
    weights = grad.mean(dim=(1, 2))                          # importance per channel
    cam = F.relu((weights[:, None, None] * act).sum(0))
    cam = cam / (cam.max() + 1e-8)
    cam = F.interpolate(cam[None, None], size=x.shape[-2:], mode="bilinear")[0, 0]
    return cam.detach().cpu().numpy()


def predict_image(image_path):
    ckpt = torch.load(OUT_DIR / "best_model.pt", map_location=DEVICE)
    model = ChestNetCNN(IN_CHANNELS, NUM_CLASSES).to(DEVICE)
    model.load_state_dict(ckpt["model_state"]); model.eval()

    thr_path = OUT_DIR / "thresholds.npy"
    thresholds = np.load(thr_path) if thr_path.exists() else np.full(NUM_CLASSES, 0.5)

    img = Image.open(image_path).convert("L")
    x = get_transforms(False, ckpt["mean"], ckpt["std"])(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probs = torch.sigmoid(model(x))[0].cpu().numpy()

    order = np.argsort(probs)[::-1]
    print("🩻 Predictions:")
    for i in order:
        flag = "✅ POSITIVE" if probs[i] >= thresholds[i] else ""
        print(f"  {CLASSES[i]:20s} {probs[i]:.3f} (thr {thresholds[i]:.2f}) {flag}")

    top = order[0]
    cam = grad_cam(model, x.clone().requires_grad_(True), top)
    base = np.array(img.resize((IMG_SIZE, IMG_SIZE)))
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1); plt.imshow(base, cmap="gray"); plt.title("Input"); plt.axis("off")
    plt.subplot(1, 2, 2); plt.imshow(base, cmap="gray"); plt.imshow(cam, cmap="jet", alpha=0.4)
    plt.title(f"Grad-CAM: {CLASSES[top]} ({probs[top]:.2f})"); plt.axis("off")
    plt.tight_layout(); plt.savefig(OUT_DIR / "gradcam.png", dpi=120); plt.show()


# Pick any test image
example = pd.read_csv(SPLIT_DIR / "test.csv")["Image Index"].iloc[0]
predict_image(IMAGE_DIR / example)

## ✅ Summary
- Resized images → patient-level split → train-only normalization → sanity check → trained a **from-scratch residual CNN** → evaluated on test (AUC, precision, recall, specificity, F1, confusion matrices) → Grad-CAM.
- 📁 Outputs are saved in `outputs/custom_cnn/`: `best_model.pt`, `history.csv`, `training_curves.png`, `test_report.csv`, `roc_curves.png`, `confusion_matrices.png`, `gradcam.png`.